In [ ]:
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════╗
║   ZERF TRANSCRIPTOR - FASE 1: TRANSCRIPCIÓN EN COLAB   ║
║   Motor: Stable-Whisper 'large' + Alineación con VTT   ║
╚══════════════════════════════════════════════════════════╝
Pega todo esto en una celda de Google Colab y ejecútala.
El modelo solo se carga una vez; si re-ejecutas la celda,
reutiliza el modelo ya cargado en memoria.
"""

# ── PARÁMETROS (ajusta aquí antes de ejecutar) ──────────────
NUM_VIDEOS             = 1    # Cuántos vídeos procesar (0 = todos los pendientes)
FORZAR_RETRANSCRIPCION = False  # True → re-transcribir aunque ya tenga SRT
VIDEO_ESPECIFICO       = ""     # ID YouTube o parte del nombre (ej: "dQw4w9WgXcQ")
                                # Si se rellena, procesa SOLO ese vídeo

# ── RUTAS EN DRIVE ───────────────────────────────────────────
DRIVE_BASE   = "/content/drive/MyDrive/Transcripts_Barca"
FOLDER_AUDIO = DRIVE_BASE + "/AUDIO_MP3"
FOLDER_SRT   = DRIVE_BASE + "/SRT_YouTube"
FOLDER_VTT   = DRIVE_BASE + "/vtt"

# ────────────────────────────────────────────────────────────
# 1. DEPENDENCIAS
# ────────────────────────────────────────────────────────────
import subprocess, sys, os

print("Instalando dependencias...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "stable-ts", "requests"])
subprocess.check_call(["apt-get", "install", "-qq", "-y", "ffmpeg"])

import re, difflib, html, gc, shutil, threading, json, requests
import torch
import stable_whisper
from datetime import datetime
from google.colab import drive

# ────────────────────────────────────────────────────────────
# 2. MONTAR DRIVE Y DESCARGAR DICCIONARIO
# ────────────────────────────────────────────────────────────
print("Montando Google Drive...")
drive.mount('/content/drive', force_remount=True)
os.makedirs(FOLDER_SRT, exist_ok=True)
print(f"  Audio MP3 : {len([f for f in os.listdir(FOLDER_AUDIO) if f.endswith('.mp3')])} archivos")
print(f"  SRTs prev : {len([f for f in os.listdir(FOLDER_SRT)   if f.endswith('.srt')])} archivos")

print("Descargando diccionario más reciente desde GitHub...")
url_diccionario = "https://raw.githubusercontent.com/Zerf8/zerf_transcriptor/main/data/diccionario.json"
try:
    resp = requests.get(url_diccionario)
    resp.raise_for_status()
    DICCIONARIO = resp.json()
    print("  Diccionario cargado correctamente.")
except Exception as e:
    print(f"  [AVISO] No se pudo cargar el diccionario de GitHub: {e}")
    DICCIONARIO = {}

# ────────────────────────────────────────────────────────────
# 3. CARGAR MODELO (solo una vez por sesión)
# ────────────────────────────────────────────────────────────
PLANTILLAS_POR_ANO = {
    2016: ("Ter Stegen, Bravo, Piqué, Mascherano, Mathieu, Digne, Jordi Alba, "
           "Aleix Vidal, Busquets, Iniesta, Rakitić, Rafinha Alcântara, "
           "Arda Turan, Denis Suárez, Messi, Suárez, Neymar, Munir"),
    2017: ("Ter Stegen, Piqué, Umtiti, Mascherano, Jordi Alba, Semedo, "
           "Busquets, Iniesta, Rakitić, André Gomes, Denis Suárez, Paulinho, "
           "Messi, Suárez, Dembélé, Deulofeu"),
    2018: ("Ter Stegen, Piqué, Lenglet, Umtiti, Jordi Alba, Semedo, "
           "Busquets, Iniesta, Rakitić, Arthur, Coutinho, "
           "Messi, Suárez, Dembélé, Malcom, Munir"),
    2019: ("Ter Stegen, Piqué, Lenglet, Umtiti, Jordi Alba, Semedo, "
           "Busquets, de Jong, Rakitić, Vidal, Arthur, Griezmann, "
           "Suárez, Messi, Dembélé, Coutinho, Junior Firpo"),
    2020: ("Ter Stegen, Piqué, Lenglet, Araujo, Jordi Alba, Dest, "
           "Busquets, de Jong, Pedri, Trincão, Mingueza, "
           "Griezmann, Messi, Dembélé, Coutinho, Braithwaite"),
    2021: ("Ter Stegen, Piqué, Araujo, Mingueza, Eric García, Dest, "
           "Jordi Alba, Busquets, de Jong, Pedri, Gavi, "
           "Luuk de Jong, Kun Agüero, Depay, Ansu Fati, "
           "Dembélé, Braithwaite, Coutinho"),
    2022: ("Ter Stegen, Piqué, Araujo, Koundé, Christensen, "
           "Balde, Jordi Alba, Azpilicueta, Marcos Alonso, "
           "Busquets, de Jong, Pedri, Gavi, "
           "Lewandowski, Raphinha, Ansu Fati, Ferran Torres, "
           "Depay, Aubameyang, Dembélé"),
    2023: ("Ter Stegen, Araujo, Koundé, Christensen, Eric García, "
           "Balde, Cancelo, Busquets, de Jong, Pedri, Gavi, "
           "Gündoğan, Fermín, Lewandowski, Raphinha, "
           "Ansu Fati, Ferran Torres, Dembélé, "
           "João Félix, Vitor Roque, Oriol Romeu"),
    2024: ("Ter Stegen, Iñaki Peña, Araujo, Koundé, Cubarsí, "
           "Eric García, Balde, Héctor Fort, Casadó, de Jong, "
           "Pedri, Gavi, Fermín, Dani Olmo, Lamine Yamal, "
           "Raphinha, Lewandowski, Pau Víctor, Ansu Fati, "
           "Flick, Hansi Flick, Laporta, Deco"),
    2025: ("Ter Stegen, Iñaki Peña, Araujo, Koundé, Cubarsí, Christensen, "
           "Balde, Héctor Fort, Álvaro Carreras, Casadó, de Jong, "
           "Pedri, Gavi, Fermín, Dani Olmo, Lamine Yamal, Marc Bernal, "
           "Raphinha, Lewandowski, Pau Víctor, Ansu Fati, "
           "Flick, Hansi Flick, Laporta, Deco"),
}
PROMPT_BASE = (
    "Transcripción de análisis del FC Barcelona en catalán y castellano. "
    "Canal: Zerf. Comunidad: Zerfistas. Culerada. "
    "Evitar repeticiones y alucinaciones. Si hay silencio, no inventar palabras. "
)

def get_prompt_for_year(filename: str) -> str:
    try:
        year = int(filename[:4])
    except (ValueError, IndexError):
        year = 2024
    available = sorted(PLANTILLAS_POR_ANO.keys())
    best = available[0]
    for y in available:
        if y <= year:
            best = y
    jugadores = PLANTILLAS_POR_ANO[best]
    return PROMPT_BASE + f"Jugadores de la temporada {best}: {jugadores}."

if 'model' not in globals() or globals().get('model') is None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nCargando modelo Stable-Whisper 'large' en {device.upper()}...")
    model = stable_whisper.load_model("large", device=device)
    print("Modelo listo." + (" (GPU)" if device == "cuda" else " (CPU - sera lento)"))
else:
    print("\nModelo ya estaba cargado. Reutilizando.")

# ────────────────────────────────────────────────────────────
# 4. FUNCIONES DE ALINEACIÓN VTT + WHISPER Y DICCIONARIO
# ────────────────────────────────────────────────────────────

def apply_dictionary_to_srt(srt_path):
    """Aplica las correcciones del diccionario (nombres, expresiones) sobre el archivo SRT."""
    if not DICCIONARIO: return

    with open(srt_path, 'r', encoding='utf-8') as f:
        content = f.read()

    correcciones = {}
    if "nombres_propios" in DICCIONARIO:
        for correcto, variaciones in DICCIONARIO["nombres_propios"].items():
            if isinstance(variaciones, list) and len(variaciones) > 0:
                correcciones[variaciones[0]] = variaciones[0]

    if "correcciones_aprendidas" in DICCIONARIO:
        for mal, bien in DICCIONARIO["correcciones_aprendidas"].items():
            correcciones[mal] = bien

    sorted_keys = sorted(correcciones.keys(), key=len, reverse=True)

    for mala_palabra in sorted_keys:
        buena_palabra = correcciones[mala_palabra]
        if not mala_palabra.strip(): continue

        pattern = re.compile(r'\b' + re.escape(mala_palabra) + r'\b', re.IGNORECASE)
        content = pattern.sub(buena_palabra, content)

    with open(srt_path, 'w', encoding='utf-8') as f:
        f.write(content)

def ts_to_ms(ts):
    ts = ts.strip().replace(',', '.')
    parts = ts.split(':')
    h, m, s = (parts if len(parts) == 3 else ['0'] + parts)
    secs, ms_part = (s.split('.') + ['0'])[:2]
    return int(h)*3600000 + int(m)*60000 + int(secs)*1000 + int(ms_part[:3].ljust(3,'0'))

def ms_to_srt(ms):
    h=ms//3600000; ms%=3600000; m=ms//60000; ms%=60000; s=ms//1000; ms%=1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def clean_vtt_line(raw):
    t = re.sub(r'<\d{1,2}:\d{2}:\d{2}\.\d{3}>', '', raw)
    t = re.sub(r'<[^>]+>', '', t)
    t = html.unescape(t)
    t = re.sub(r'\[\s*__\s*\]', '', t)
    return ' '.join(t.split())

def is_desc_only(text):
    return len(re.sub(r'\[[^\]]*\]', '', text).strip()) == 0 and len(text.strip()) > 0

def parse_vtt(vtt_path):
    with open(vtt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    segments = []
    for block in re.split(r'\n\n+', content):
        lines = block.strip().splitlines()
        m = None
        for i, line in enumerate(lines):
            m = re.match(r'(\d{1,2}:\d{2}:\d{2}[.,]\d{3})\s*-->\s*(\d{1,2}:\d{2}:\d{2}[.,]\d{3})', line)
            if m:
                text_lines = lines[i+1:]; break
        if not m: continue
        start_ms, end_ms = ts_to_ms(m.group(1)), ts_to_ms(m.group(2))
        if end_ms - start_ms < 100: continue
        useful = [l for l in text_lines if l.strip() and l.strip() not in ('\xa0',' ')]
        if not useful: continue
        clean = clean_vtt_line(useful[-1])
        if not clean: continue
        segments.append({'start_ms':start_ms,'end_ms':end_ms,'text':clean,
                         'is_desc':is_desc_only(clean),'assigned_words':[]})
    return segments

def normalize(w):
    return re.sub(r'[^\w]','',w.lower())

def align_vtt_whisper(segments, whisper_srt_path):
    with open(whisper_srt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    content = re.sub(r'\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*\d{2}:\d{2}:\d{2}[,.]\d{3}','',content)
    content = re.sub(r'^\d+\s*$','',content,flags=re.MULTILINE)
    wh_words = content.split()
    for seg in segments: seg['assigned_words'] = []
    yt_words, yt_idx = [], []
    for i, seg in enumerate(segments):
        if seg['is_desc']: seg['assigned_words']=[seg['text']]; continue
        for w in seg['text'].split():
            if w: yt_words.append(w); yt_idx.append(i)
    if not yt_words or not wh_words: return segments
    matcher = difflib.SequenceMatcher(None,[normalize(w) for w in yt_words],[normalize(w) for w in wh_words],autojunk=False)
    for tag,i1,i2,j1,j2 in matcher.get_opcodes():
        if tag=='equal':
            for k in range(j2-j1): segments[yt_idx[i1+k]]['assigned_words'].append(wh_words[j1+k])
        elif tag=='replace':
            affected=[]
            for i in range(i1,i2):
                if i<len(yt_idx):
                    s=yt_idx[i]
                    if not affected or affected[-1]!=s: affected.append(s)
            if affected: segments[affected[0]]['assigned_words'].extend(wh_words[j1:j2])
            elif segments: segments[-1]['assigned_words'].extend(wh_words[j1:j2])
        elif tag=='insert':
            t=min(i1,len(yt_idx)-1) if yt_idx else 0
            if yt_idx: segments[yt_idx[t]]['assigned_words'].extend(wh_words[j1:j2])
    return segments

def write_aligned_srt(segments, out_path):
    idx=1
    with open(out_path,'w',encoding='utf-8') as f:
        for seg in segments:
            text = seg['text'] if seg['is_desc'] else ' '.join(seg['assigned_words']).strip()
            if not text: continue
            f.write(f"{idx}\n{ms_to_srt(seg['start_ms'])} --> {ms_to_srt(seg['end_ms'])}\n{text}\n\n")
            idx+=1
    return idx-1

def get_video_id(filename):
    m = re.search(r'([A-Za-z0-9_-]{11})\.(mp3|srt|vtt)$', filename)
    return m.group(1) if m else None

def find_vtt(video_id, folder):
    for f in os.listdir(folder):
        if video_id in f and f.endswith('.vtt'):
            return os.path.join(folder, f)
    return None

# ────────────────────────────────────────────────────────────
# 5. BUCLE PRINCIPAL
# ────────────────────────────────────────────────────────────
todos   = sorted(f for f in os.listdir(FOLDER_AUDIO) if f.endswith('.mp3'))
srts_ok = set(f for f in os.listdir(FOLDER_SRT) if f.endswith('.srt'))

pendientes = []
for audio in todos:
    vid = get_video_id(audio)
    if not vid: continue
    # Modo vídeo específico: procesar solo el que coincida
    if VIDEO_ESPECIFICO:
        if VIDEO_ESPECIFICO.lower() in audio.lower():
            pendientes.append(audio)
        continue
    ya_tiene = any(vid in s for s in srts_ok)
    if ya_tiene and not FORZAR_RETRANSCRIPCION: continue
    pendientes.append(audio)

total = len(pendientes)
if NUM_VIDEOS > 0:
    pendientes = pendientes[:NUM_VIDEOS]

print(f"\nAudios totales:  {len(todos)}")
print(f"Pendientes:      {total}  |  Procesando ahora: {len(pendientes)}\n")

upload_thread = None  # hilo de subida a Drive del vídeo anterior

for n, audio_file in enumerate(pendientes, 1):
    # Esperar a que termine la subida del SRT anterior antes de empezar
    if upload_thread and upload_thread.is_alive():
        print("  [⏳ esperando subida anterior...]")
        upload_thread.join()

    t0 = datetime.now()
    audio_path = os.path.join(FOLDER_AUDIO, audio_file)
    base       = audio_file[:-4]
    vid        = get_video_id(audio_file)
    srt_drive  = os.path.join(FOLDER_SRT, f"{base}.srt")  # destino final en Drive
    srt_local  = f"/tmp/{base}_final.srt"                  # SRT final en local
    srt_tmp    = f"/tmp/{base}_whisper.srt"                # SRT bruto de Whisper

    print(f"[{n}/{len(pendientes)}] {base[:72]}")
    print(f"  Inicio: {t0.strftime('%H:%M:%S')}")

    try:
        # 1. Copiar MP3 a /tmp para lectura local rápida
        mp3_local = f"/tmp/{audio_file}"
        shutil.copy(audio_path, mp3_local)

        # 2. Transcribir desde /tmp (sin latencia de Drive)
        prompt = get_prompt_for_year(audio_file)
        print(f"  Prompt año {audio_file[:4]}: {len(prompt)} chars")
        result = model.transcribe(
            mp3_local, language="es",
            initial_prompt=prompt,
            vad=True, regroup=True, word_timestamps=True,
        )
        os.remove(mp3_local)  # liberar espacio en /tmp
        result.split_by_length(max_chars=42, max_words=12).to_srt_vtt(
            srt_tmp, word_level=False, segment_level=True
        )

        apply_dictionary_to_srt(srt_tmp)

        # 3. Alinear con VTT → guardar en local (/tmp)
        vtt_path = find_vtt(vid, FOLDER_VTT) if vid else None
        if vtt_path:
            print(f"  Alineando con VTT: {os.path.basename(vtt_path)}")
            segs = parse_vtt(vtt_path)
            if segs:
                aligned = align_vtt_whisper(segs, srt_tmp)
                n_bloques = write_aligned_srt(aligned, srt_local)
                print(f"  SRT alineado: {n_bloques} bloques")
            else:
                shutil.copy(srt_tmp, srt_local)
                print("  VTT vacio → usando Whisper directo")
        else:
            shutil.copy(srt_tmp, srt_local)
            print("  Sin VTT → usando Whisper directo")

        if os.path.exists(srt_tmp): os.remove(srt_tmp)

        # 4. Subir SRT a Drive en background mientras procesamos el siguiente
        def _subir_a_drive(src, dst):
            try:
                shutil.copy(src, dst)
                os.remove(src)
            except Exception as ex:
                print(f"  [ERROR subida] {ex}")

        upload_thread = threading.Thread(
            target=_subir_a_drive, args=(srt_local, srt_drive), daemon=True
        )
        upload_thread.start()

        t1 = datetime.now()
        print(f"  Fin:    {t1.strftime('%H:%M:%S')}  (duración: {str(t1-t0).split('.')[0]})")
        print(f"  ↗ Subiendo a Drive en background: {os.path.basename(srt_drive)}\n")

    except Exception as e:
        import traceback; print(f"  ERROR: {e}\n{traceback.format_exc()}")

    finally:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# Esperar a que termine la última subida pendiente
if upload_thread and upload_thread.is_alive():
    print("Esperando última subida a Drive...")
    upload_thread.join()

print(f"\n=== FIN ===  SRTs en Drive: {len([f for f in os.listdir(FOLDER_SRT) if f.endswith('.srt')])}")